#  ongoing orders without paid invoices in the current fiscal year 

In [1]:
import pandas as pd
import requests
from datetime import datetime

pd.set_option('display.max_columns', None)

## 1. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [2]:
%run folio_auth.ipynb

Login succeeded. Token retrieved.


## Logic
- Get all POs with a status of open and orderType of Ongoing
- Get all POLs with receiptStatus of Ongoing
- Get all Invoices with an Invoice Date of the current FY
- Merge and highlight those POLs with no invoice date or invoice paid date for the current FY. --> This could capture invoices created, but not approved/paid.


In [3]:
def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit
    return all_records

In [4]:
fys_raw = fetch_all_records(
    "/finance/fiscal-years",
    records_key="fiscalYears"
) 
fys_df = pd.DataFrame(fys_raw)
today = datetime.today().strftime("%Y-%m-%d")
cur_fy_df = fys_df[(fys_df['periodStart'] <= today) & (fys_df['periodEnd'] >= today)]
styled_fy = cur_fy_df.style.set_properties(
  **{'border': '1px solid black', 'background-color': 'lightgrey'}  
)
print(f"Fiscal Years that encompass {today}")
cur_fy_df[
    ['id', 'name', 'periodStart', 'periodEnd']
].style.set_properties(
    **{'border': '1px solid black'}
)

Fiscal Years that encompass 2026-08-27


,id,name,periodStart,periodEnd
7,d01d3d59-a8e7-4b58-a485-5146f5747dbb,Fiscal Year 2027,2026-07-01T00:00:00.000+00:00,2027-06-30T00:00:00.000+00:00
8,ab5d32df-99ec-4487-86ea-e4203143c8d3,Calendar Year 2026,2026-01-01T00:00:00.000+00:00,2026-12-31T00:00:00.000+00:00


1. Get all POs ongoing POs with a status of open and orderType of Ongoing
2. Get all POLs with receiptStatus of Ongoing
3. Get all Invoices with an Invoice Date of the current FY
4. Merge and highlight rows with no invoices

In [ ]:
orders_raw = fetch_all_records(
    "/orders/composite-orders",
    records_key="purchaseOrders",
    query='workflowStatus="Open" and orderType = "Ongoing"'
    ) 
orders_df = pd.DataFrame(orders_raw)

print(f"{len(orders_raw)} open, ongoing orders found")
orders_df.head()

,id,approved,approvedById,approvalDate,billTo,dateOrdered,manualPo,notes,poNumber,orderType,reEncumber,ongoing,shipTo,template,vendor,workflowStatus,acqUnitIds,tags,metadata,customFields,assignedTo
0,40aaa902-bfc8-4f20-af9f-e1cd729acde9,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-04-21T17:04:42.479+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-04-21T17:04:42.479+00:00,False,[],21624,Ongoing,True,"{'interval': 365, 'isSubscription': True, 'man...",4d098cbe-04cd-4332-9fad-24208fae34c4,74b9551e-f7e5-43e8-887f-7d6b075e9b19,ae62585b-5585-41a7-9519-5b16c1d32d54,Open,[a3f803ba-587c-4aac-b2fc-35226659c4fe],{'tagList': []},{'createdDate': '2025-04-21T16:59:34.873+00:00...,NaN,NaN
1,180bcb30-7b70-4536-bd0b-6b6cf231ddfa,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-05-07T04:31:04.111+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-05-07T04:31:04.111+00:00,False,[],21655,Ongoing,False,"{'interval': 365, 'isSubscription': True, 'man...",NaN,5b0bf4ea-7971-4717-a77c-8ce7f2577685,fed3982c-1e92-4719-bb9b-42129018fd25,Open,[],{'tagList': []},{'createdDate': '2025-05-07T04:29:23.084+00:00...,NaN,NaN
2,e02be3d3-63b4-45a8-a82d-d8e001adb813,False,7c2b76f7-634e-47d1-9aa5-9910751f23c8,2025-05-09T11:10:06.741+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-05-09T11:10:06.741+00:00,NaN,[],21660,Ongoing,True,"{'interval': 1, 'isSubscription': True, 'manua...",4d098cbe-04cd-4332-9fad-24208fae34c4,NaN,1dc0f2dc-91b1-41bb-bb7f-145815385366,Open,[],NaN,{'createdDate': '2025-05-09T11:06:25.857+00:00...,NaN,NaN
3,f74c9128-0666-43c9-8698-0eeea8dfc036,False,91d619a5-66b5-45f3-8680-42525ec90bcd,2025-05-29T12:00:20.752+00:00,NaN,2025-05-29T12:00:20.752+00:00,False,[],21712,Ongoing,False,"{'interval': 12, 'isSubscription': True, 'manu...",NaN,79d99241-f68e-474e-a104-da82219edb9e,1ce90497-a8ba-4940-9ffc-0044793e91a0,Open,[5f3bbae0-5d37-460f-a47f-71224c2952a5],{'tagList': []},{'createdDate': '2025-05-29T11:57:43.165+00:00...,NaN,NaN
4,4fa80b0a-1934-4f90-ba1e-8dcaf208ec29,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-06-12T19:59:04.021+00:00,NaN,2025-06-12T19:59:04.021+00:00,False,[],21738,Ongoing,False,"{'isSubscription': False, 'manualRenewal': False}",NaN,15b8c257-7438-4937-9630-d57d2b679cb4,a431ca52-328c-4a97-880a-3689841faba6,Open,[],{'tagList': []},{'createdDate': '2025-06-12T17:18:41.319+00:00...,NaN,NaN
